# De la exploración al reporte de mediciones

La primera notebook nos permitió comprobar formas, seleccionar rondas y resumir
columnas. Ahora usaremos esas operaciones como funciones reutilizables desde la
notebook y desde `main.py`, sin reconstruir la lógica en cada celda.

## Contenido

1. Cargar y validar la forma.
2. Seleccionar rondas completas.
3. Resumir por sala e interpretar el resultado.
4. Añadir una selección propia.

Abre esta notebook en VS Code desde `02-reporte-mediciones` y selecciona el kernel de `.venv`.


In [1]:
import json
from pathlib import Path

import numpy as np

from measurements.processing import (
    ROOM_NAMES,
    load_readings,
    select_complete_rows,
    summarize,
)

project_dir = Path.cwd()
data_dir = project_dir.parent / "datos"
if not (project_dir / "main.py").is_file() or not data_dir.is_dir():
    raise FileNotFoundError(
        "Start the notebook from the 02-reporte-mediciones project directory"
    )

## 1. Cargar y validar la forma

`load_readings` comprueba el encabezado y lee una matriz con tres columnas.
La comprobación impide asignar un resumen a otra sala si cambia el orden del CSV.
Abre el módulo en el editor e identifica qué parte lee y cuál valida.

[Lectura con loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [2]:
readings = load_readings(data_dir / "readings.csv")
print(ROOM_NAMES)
print(readings.shape)
print(readings[:2])
assert readings.shape == (8, 3)

('north_room', 'south_room', 'storage_room')
(8, 3)
[[22. 24. 20.]
 [23. 25. 21.]]


## 2. Seleccionar rondas completas

La función devuelve dos resultados: las filas seleccionadas y la cantidad de
rechazos. Esto permite conservar evidencia del filtrado. Ni el arreglo original
ni el CSV se modifican.

In [3]:
quality_readings = load_readings(data_dir / "readings_quality.csv")
original = quality_readings.copy()
complete, rejected = select_complete_rows(quality_readings)
print(complete)
print(f"Rejected rows: {rejected}")
assert rejected == 2
assert not np.shares_memory(complete, quality_readings)
np.testing.assert_array_equal(quality_readings, original)

[[22. 24. 20.]
 [24. 26. 22.]]
Rejected rows: 2


### Comprobación: ninguna ronda completa

No calcularemos medias sobre una matriz vacía. La función comunica el problema
con una excepción; la notebook puede capturarla para discutir el caso y continuar.

In [4]:
invalid_readings = np.array([[np.nan, 24, 20], [22, np.inf, 21]])
try:
    select_complete_rows(invalid_readings)
except ValueError as error:
    print(f"Expected error: {error}")
else:
    raise AssertionError("Expected an error for data without complete rows")

Expected error: No complete readings remain


## 3. Resumir por sala

Las operaciones numéricas se aplican por eje. El pequeño ciclo de la función
`summarize` asigna nombres a los tres resultados y convierte los escalares a
`float` de Python para serializarlos en JSON. No recorre las mediciones para
calcular sus medias.

In [5]:
report = summarize(complete)
print(json.dumps(report, indent=2, allow_nan=False))
np.testing.assert_allclose(
    [report[name]["mean_c"] for name in ROOM_NAMES], [23, 25, 21]
)

{
  "north_room": {
    "mean_c": 23.0,
    "min_c": 22.0,
    "max_c": 24.0
  },
  "south_room": {
    "mean_c": 25.0,
    "min_c": 24.0,
    "max_c": 26.0
  },
  "storage_room": {
    "mean_c": 21.0,
    "min_c": 20.0,
    "max_c": 22.0
  }
}


### Ejercicio 10: explicar el cambio de medias

Compara el reporte de las ocho rondas originales con el de las dos rondas
completas del archivo con errores. Describe qué datos entraron en cada cálculo.
No son dos momentos de una instalación real: son archivos de práctica distintos.

In [6]:
# El archivo original también pasa primero por la selección de filas completas.
original_complete, original_rejected = select_complete_rows(readings)
assert original_rejected == 0
assert original_complete.shape == (8, 3)
assert complete.shape == (2, 3)
assert rejected == 2
assert np.array_equal(complete, [[22, 24, 20], [24, 26, 22]])

original_report = summarize(original_complete)
original_means = np.array([original_report[name]["mean_c"] for name in ROOM_NAMES])
quality_means = np.array([report[name]["mean_c"] for name in ROOM_NAMES])
assert np.allclose(original_means, [25.5, 27.5, 23.5])
assert np.allclose(quality_means, [23, 25, 21])
assert np.allclose(quality_means - original_means, [-2.5, -2.5, -2.5])
print("Salas:", ROOM_NAMES)
print("Medias del archivo original (8 rondas):", original_means)
print("Medias del archivo con errores (2 rondas completas):", quality_means)

Salas: ('north_room', 'south_room', 'storage_room')
Medias del archivo original (8 rondas): [25.5 27.5 23.5]
Medias del archivo con errores (2 rondas completas): [23. 25. 21.]


**Interpretación del ejercicio 10.** En `readings.csv` entraron las ocho rondas
(índices 0–7), con sus tres temperaturas, y ninguna se rechazó. Sus medias son
`[25.5, 27.5, 23.5]` °C en el orden norte, sur y almacén.
En `readings_quality.csv` entraron únicamente las filas 0 y 2:
`[22, 24, 20]` y `[24, 26, 22]`. Se descartaron enteras las filas 1 y 3 por
contener `nan` e `inf`, incluidas las temperaturas válidas de sus otras salas.
Las dos rondas aceptadas dan `[23, 25, 21]` °C, 2.5 °C menos en cada sala.
Cada media usa las mismas rondas en las tres columnas. La diferencia corresponde
a los valores que contiene cada archivo de práctica; no demuestra un cambio
temporal de una instalación real.

## 4. Añadir una selección propia

### Ejercicio 11: seleccionar rondas cálidas

Resuelve la [práctica](../PRACTICA.md) creando `select_warm_rounds` en un módulo
nuevo. Reinicia el kernel después de guardar el módulo y comprueba aquí las filas
seleccionadas mediante `assert`.

Para umbral `27`, deben quedar tres rondas del archivo original. Para umbral
`100`, la selección es vacía y no se debe calcular su media.

In [7]:
from measurements.selection import select_warm_rounds

# original_complete contiene las ocho filas finitas de readings.csv.
threshold = 27
warm_readings = select_warm_rounds(original_complete, threshold)
expected_warm = np.array([[27, 29, 25], [28, 30, 26], [29, 31, 27]])
assert warm_readings.shape == (3, 3)
assert np.array_equal(warm_readings, expected_warm)
# El umbral es inclusivo: la ronda con temperatura norte de 27 sí entra.
assert warm_readings[0, 0] == threshold
assert np.all(warm_readings[:, 0] >= threshold)
print("Rondas con temperatura norte >= 27 °C:")
print(warm_readings)

Rondas con temperatura norte >= 27 °C:
[[27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]


La comparación usa solo la columna norte para decidir qué **filas completas**
conservar; cada fila seleccionada mantiene las columnas norte, sur y almacén.
Con umbral `100` no hay coincidencias. Una matriz de forma `(0, 3)` conserva la
estructura de las salas, pero tiene cero rondas: su media no está definida porque
no hay valores que sumar y dividir entre una cantidad positiva. No llamamos a
`mean` ni a `summarize` para esa selección, evitando resultados `nan` y avisos.

In [8]:
no_warm_readings = select_warm_rounds(original_complete, 100)
assert no_warm_readings.shape == (0, 3)
assert no_warm_readings.size == 0
print("Selección con umbral 100:", no_warm_readings.shape)

Selección con umbral 100: (0, 3)


La función exige una entrada no vacía con tres columnas y valores finitos en
todas las salas, además de un umbral finito. Las pruebas siguientes introducen
`nan`, `inf` y `-inf` en cada sala y como umbrales: todos deben producir
`ValueError`, incluso si la fila con el dato inválido no superaría el umbral.

In [9]:
for nonfinite in (np.nan, np.inf, -np.inf):
    for column in range(3):
        invalid = original_complete.copy()
        invalid[0, column] = nonfinite
        try:
            select_warm_rounds(invalid, 27)
        except ValueError as error:
            print(f"Error esperado para {nonfinite} en columna {column}: {error}")
        else:
            raise AssertionError("Se esperaba ValueError para datos no finitos")

    try:
        select_warm_rounds(original_complete, nonfinite)
    except ValueError as error:
        print(f"Error esperado para umbral {nonfinite}: {error}")
    else:
        raise AssertionError("Se esperaba ValueError para un umbral no finito")

Error esperado para nan en columna 0: Selection requires finite readings
Error esperado para nan en columna 1: Selection requires finite readings
Error esperado para nan en columna 2: Selection requires finite readings
Error esperado para umbral nan: Selection requires a finite threshold
Error esperado para inf en columna 0: Selection requires finite readings
Error esperado para inf en columna 1: Selection requires finite readings
Error esperado para inf en columna 2: Selection requires finite readings
Error esperado para umbral inf: Selection requires a finite threshold
Error esperado para -inf en columna 0: Selection requires finite readings
Error esperado para -inf en columna 1: Selection requires finite readings
Error esperado para -inf en columna 2: Selection requires finite readings
Error esperado para umbral -inf: Selection requires a finite threshold


In [10]:
original_before_selection = original_complete.copy()
readings_before_selection = readings.copy()
editable_selection = select_warm_rounds(original_complete, 27)
assert not np.shares_memory(editable_selection, original_complete)
editable_selection[0, :] = 100
assert np.array_equal(editable_selection[0], [100, 100, 100])
assert np.array_equal(original_complete, original_before_selection)
assert np.array_equal(readings, readings_before_selection)
assert np.array_equal(warm_readings, expected_warm)
print("La selección se modificó sin cambiar las ocho rondas originales.")

La selección se modificó sin cambiar las ocho rondas originales.


In [11]:
warm_report = summarize(warm_readings)
warm_means = np.array([warm_report[name]["mean_c"] for name in ROOM_NAMES])
assert np.allclose(warm_means, [28, 30, 26])
assert np.allclose(original_means, [25.5, 27.5, 23.5])
assert np.allclose(warm_means - original_means, [2.5, 2.5, 2.5])
print("Medias de las 3 rondas seleccionadas:", warm_means)
print("Medias de las 8 rondas originales:", original_means)

Medias de las 3 rondas seleccionadas: [28. 30. 26.]
Medias de las 8 rondas originales: [25.5 27.5 23.5]


**Interpretación del ejercicio 11.** La selección con umbral `27` incluye las
filas 5, 6 y 7 del archivo original: `[27, 29, 25]`, `[28, 30, 26]` y
`[29, 31, 27]`. Cada media de la selección usa esas mismas tres rondas y resulta
en `[28, 30, 26]` °C. El conjunto completo usa las ocho filas originales,
con medias `[25.5, 27.5, 23.5]` °C. Las medias seleccionadas son 2.5 °C mayores
en cada sala porque aquí elegimos las últimas tres rondas, que tienen valores
más altos en las tres columnas. La condición se aplicó únicamente a la sala
norte; el aumento en las otras salas se observa en estos datos, sin que la
función imponga una condición sobre sus temperaturas.
La selección editable es independiente: cambiarla no altera la entrada ni la
selección usada para calcular las medias.

## Verificación final

Desde la terminal del proyecto ejecuta `uv run --locked python main.py` para
generar los archivos. Después reinicia el kernel y ejecuta toda la notebook.
Las pruebas existentes con pytest son una comprobación opcional.
